# Linear Regression: Expanding the Feature Space
## DS-3001: Machine Learning 1

Content adapted from Terence Johnson (UVA)

**Notebook Summary:** In the previous notebook, we introduced linear regression with both single and multiple variables. In these models, we used either the numeric variables in the raw data or one hot encodings of categorical variables. Today, we'll talk about ways that we can increase the number of features in our model to make more complex models. We can expand the complexity of our model by creating new columns in our design matrix. With this ability to increase the feature space, we need to be cautious to understand that increasing the complexity may lead to overfitting on our training data set.

### Loading Environment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os # For changing directory

# For regression
from sklearn.linear_model import LinearRegression

# Import plotly for three dimensional plots
import plotly.express as px
import plotly.graph_objects as go

# To mount your google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path_to_DS_3001_folder = '/content/drive/MyDrive/DS-3001/02_Intro_to_ML_Algorithms'
# path_to_DS_3001_folder = ''

# Update the path to your folder for the class
# Where you stored the data from the previous noteboook
os.chdir(path_to_DS_3001_folder)

### Data Set for Today

In [ ]:
# We'll continue to work with the craigslist data set from before

# Load in the data frame
car_df = pd.read_csv('data/craiglist_cville_cars_long.csv', low_memory = False)

# Look at the first 5 rows of data frame
car_df.head(5)

# Feature Engineering

## What, exactly, makes a model linear?
- A model is linear because the coefficients, $b$, enter the prediction as
$$
\hat{y} = b \cdot \hat{x} = b_1 \hat{x}_1 + b_2 \hat{x}_2 + ... b_J \hat{x}_J
$$
so that the relationship between $\hat{y}$ and each $\hat{x}_j$ is a linear one through $b_j$

- Taking non-linear transformations of the $x_j$'s simply gives new variables, like $\log(x_j)$ or $x_j^2$ -- *Transformations of the regressors don't make the model nonlinear*

- Likewise, if there was a coefficient written in a funky nonlinear way, like $\sqrt{ \text{asinh}(b_j)}$, you can simply relabel that coefficient as $b_j \leftarrow \sqrt{ \text{asinh}(b_j)}$ and make the model linear again.

- There *are* many interesting non-linear models, but they are often derived from a domain-specific theory and require additional work to understand

## Beyond Fitting the Model
- Up to here, the discussion has been a standard analysis of linear models

- From the perspective of data science, we have **model specification** questions: What variables should go in the model?

- The power of linear models is that you can interact variables and expand the feature space, allowing you to better control how relationships between variables are modelled.

- The risk is that if you add too many features, your model reliability can degrade because of issues such as multicollinearity.

## Feature Spaces
- The data that we get is just raw material. We have to deliberately create a **feature space** that allows models to be expressive and clever about using the data.

- Already, we've seen various useful ways of processing the data: log transforms, hyperbolic arcsin transformations, maxmin scaling, one-hot encoding.

- When we create/transform new variables, we create new information that was previously unavailable, expanding the space of models that we can estimate. We're increasing the dimension of our feature space.

## Normalization/Standardization Techniques

- Especially with neighbor-based models and neural networks, normalizing the scale of the variables is important for computational stability. Below are some methods for how we can scale individual variables and when they can be useful.

- **Maxmin**: Useful when the data do not follow a normal distribution, but does not have many outliers. This allows you to retain the shape of that original distribution.
$$
u_i = \frac{x_i - \min(x)}{\max(x)-\min(x)}
$$

- **$z$-score**: If all the variables are roughly normally distributed / bell shaped (or are, after a log/ihs transformation), we can center them around 0 and give them a standard deviation of 1.
$$
z_i = \frac{x_i - \bar{x} }{s(x)}
$$

- **Robust scaling**: If there are outliers that make $z$-scaling unreliable, we can use the robsut scaling method instead
$$
r_i = \frac{x_i - \text{median}(x)}{IQR(x)}
$$

In [ ]:
# Looking at three data examples for each type of scaling
np.random.seed(42)

non_normal_no_outliers = np.random.gamma(shape=12.0, scale=2.0, size=1000)
normal_data = np.random.normal(10, 15, size = 1000)
outlier_data = np.random.standard_t(df=1, size = 1000)

# Let's look at the distribution of each
plt.boxplot(non_normal_no_outliers, vert = False)
plt.title('Non Normal with Few Outliers')
plt.show()

plt.boxplot(normal_data, vert = False)
plt.title('Normal with Few Outliers')
plt.show()

plt.boxplot(outlier_data, vert = False)
plt.title('Non Normal with Many Outliers')
plt.show()

# Create functions for each of the transformations
def MinMaxScaling(x):
  numerator = x - np.min(x)
  denominator = np.max(x)-np.min(x)
  out = numerator / denominator
  return out

def ZScoreScaling(x):
  numerator = x - np.mean(x)
  denominator = np.std(x)
  out = numerator / denominator
  return out

def RobustScaling(x):
  numerator = x - np.median(x)
  IQR = np.quantile(x, 0.75) - np.quantile(x, 0.25)
  out = numerator / IQR
  return out

# Apply these functions to the each of our data and plot the transformed versions

# MinMax Scaling
plt.boxplot(MinMaxScaling(non_normal_no_outliers), vert = False)
plt.title('Min Max Scaling')
plt.show()

plt.boxplot(ZScoreScaling(normal_data), vert = False)
plt.title('Z Score Scaling')
plt.show()

plt.boxplot(RobustScaling(np.arcsinh(outlier_data)), vert = False)
plt.title('Robust Scaling')
plt.show()


In [ ]:
# Look at the histogram and boxplot of mileage and age
sns.histplot(car_df['miles'])
plt.title('Miles Histogram')
plt.show()

sns.boxplot(car_df['miles'], orient = 'h')
plt.title('Miles Boxplot')
plt.show()

sns.histplot(car_df['age'])
plt.title('Age Histogram')
plt.show()

sns.boxplot(car_df['age'], orient = 'h')
plt.title('Age Boxplot')
plt.show()

# QUESTION: How can we scale our miles and age terms?


## Dummy Variables for Missing Values

- For regression models, the following approach works very well:

1. Create a missing value dummy, `df['var_na'] = df['var'].isna()`
2. Replace missings with zero, `df['var_0'] = df['var'].nafill(0)`
3. Including both `var_na` and `var_0` in your regression model.

Why does this work?

$$
... + b_{\text{var}_{NA}} \text{var}_{NA} + b_{\text{var}_0} \text{var}_0+... = \begin{cases}
\beta_{\text{var}_0} \text{var}_0, & \text{ data available }\\
\beta_{\text{var}_{NA}}, & \text{ data missing }
\end{cases}
$$

- So the model smoothly switches back and forth between a linear model when data are available, and a missing data dummy when it's not available.

- **Only do this with models based on linear combinations of variables**

In [ ]:
# We can implement this for the age variable in our car data set and create a regression model



## Expanding the Feature Space
- To fully leverage the power of regression, you have many options to expand the range of variables that the model can use to explain the variation in the data:
    - An **interaction term** is when you take two explanatory variables, say $x_1$ and $x_2$, and multiply them together to get a new explanatory variable, $z = x_1 x_2$, like $\text{mileage} \times \text{age}$
    - A **polynomial family** is when you take an explanatory variable, say $x$, and compute its powers $x^2$, $x^3$, ... , $x^K$. (There are many kinds of families besides polynomial.)
    - A **logarithmic transformation** or **inverse hyperbolic sine transformation**
    - A **maxmin normalization** or **z-score normalization**
    - More advanced tools like **principal components analysis** decomposition of the data or **splines** that create highly transformations of variables
    - Any combination of the above

- This is where we run into a significant danger of overfitting: The more complex the feature space, the more opportunities we give the model to pick non-representative cases around which to build non-representative/externally invalid models

#### Implementing Interactions Between Variables in Our Data Set

* Interaction is interested in looking at the nonlinear effects of interacting two or more variables at a time. This takes the form of:
$$
b_{jk} x_{j}x_{k}, \quad j \neq k
$$
for 2 variables or
$$
b_{jk\ell} x_{j}x_{k}x_{\ell}, \quad j \neq k \neq \ell
$$
for 3. This captures the joint non-linear impact of changes in $x_j$ and $x_k$ or $x_\ell$ together.

* Let's look at an example of interacting the `miles` and `age` variables in our data set.

In [ ]:
# Implement an interaction variable by hand of scaled miles and age


In [ ]:
# Does it look at like there's a higher order relationship between the miles and age variables and price?

# Miles vs Price
plt.scatter(
    X_scaled['miles_scaled'],
    X_scaled['price']
)
plt.title('Miles vs Price')
plt.show()

# Age vs Price
plt.scatter(
    X_scaled['age_scaled'],
    X_scaled['price']
)
plt.title('Age vs Price')
plt.show()

In [ ]:
# Let's create a model that has higher order terms for miles


In [ ]:
# Do the same for age


## Quick Polynomial and Interaction Features/Variables Using Sklearn
- It it tedious to compute interaction terms like $x_1 \times x_2$ or $x^2, x^3, ..., x^m$ on your own, and it's a very common task, so `sklearn` has a convenient tool for accomplishing this

- The `PolynomialFeatures` is the object in `sklearn.preprocessing` that can quickly create matrices of exponenetiated variables for you without doing it yourself

- The basic steps are:
    1. Import the object: `from sklearn.preprocessing import PolynomialFeatures`
    2. Decide the degree of the expansion and create an expander: `expander = PolynomialFeatures(degree=2,include_bias=False)`
    3. Execute the transformations: `Z = expander.fit_transform(X)` and get labels for the columns `names = expander.get_feature_names_out()`
    4. Create a new dataframe: `zdf = pd.DataFrame(data=Z, columns = names)`
    5. Possibly concatenate this new dataframe with other data
    
- Use this power wisely

In [ ]:
# Implement interaction variable using Skelarn of miles and age
from sklearn.preprocessing import PolynomialFeatures


In [ ]:
# Implement higher degree terms up to degree 5 for Sklearn using miles


## Quickly Concatenating Dataframes

- After making each of these new features of your design matrix, you need a way to get them back into a single dataframe. The best way to do this is via concatenation in Pandas.

- The `df = pd.concat([df1,df2,...,dfk],axis=1)` makes a new dataframe out of the columns of the original dataframes `df1`, `df2`, ..., `dfk`

- So you can build all the transformed data frames you want, then collapse them all back into one dataframe for processing.

- Since you are going to do this, you might want to think carefully about how you're building the dataframe chunks, and use `.iloc` or `.loc` to be selective about what going into each one.

In [ ]:
# Create a combined design matrix that has
# 1. polynomial degrees up to 3 for miles
# 2. Polynomila degrees up to 5 for age
# 3. a one hot encoding of color


## Proceed with Caution When Increasing the Complexity of Your Design Matrix

* If we increase the complexity of our design matrix, we run into the potential of overfitting to the training data.

* The more complex the feature space, the more opportunities we give the model to pick non-representative cases around which to build non-representative/externally invalid models

* These overfit models will not generalize to unseen data, thus performing worse on our test data set.

* Let's look at an example of overfitting with polynomial terms for mileage.

In [ ]:
# Do a train test split of the data frame
from sklearn.model_selection import train_test_split # Creating a train test split

# Function for MSE
def mse(y_true, y_hat):

  # Calculate the squared errors
  squared_errors = (y_hat - y_true)**2

  # Calculate the mean of the squared errors
  mean_squared_errors = np.sum(squared_errors) / len(y_true)

  return mean_squared_errors

seed = 90

# Isolating the miles scaled and price
X = X_scaled[['miles_scaled']]
y = X_scaled['price']

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.5, # Use a small value of the test size since we don't have many observations
    random_state = seed
)

# Investigate different powers
degrees = np.arange(1, 16)
train_mse = []
test_mse = []

# Loop over degrees
for degree in degrees:

  # Create the features for the design matrix
  expander = PolynomialFeatures(degree=degree, include_bias=False)

  # Apply the expander to our train and test data
  Z_train = expander.fit_transform(X_train[['miles_scaled']])
  Z_test = expander.transform(X_test[['miles_scaled']])  # use transform instead of fit_transform because we want to use the transformation we made on the train data set

  # Get the names of these variables
  names = expander.get_feature_names_out()

  # Create a new, expanded dataframe for the train and test data sets
  X_train_design_mat = pd.DataFrame(data=Z_train, columns = names)
  X_test_design_mat = pd.DataFrame(data=Z_test, columns = names)

  # Fit a linear regression model on the train data design matrix
  model = LinearRegression()
  model = model.fit(X_train_design_mat, y_train)

  # Get predictions on the train and test data sets
  y_hat_train = model.predict(X_train_design_mat)
  y_hat_test = model.predict(X_test_design_mat)

  # Calculate the mse for the train and test data sets
  mse_train = mse(y_train, y_hat_train)
  mse_test = mse(y_test, y_hat_test)

  # Append to the lists to save the values of the train and test mse
  train_mse.append(mse_train)
  test_mse.append(mse_test)

# Plot the results of the train and test mse for different degrees
plt.plot(
    degrees,
    train_mse,
    label = 'Train MSE',
    color = 'firebrick',
    marker = 'o'
)

plt.plot(
    degrees,
    test_mse,
    label = 'Test MSE',
    color = 'dodgerblue',
    marker = 'o'
)

plt.xlabel('Highest Polynomial Degree Included in the Model')
plt.ylabel('Mean Squared Error (MSE) of Predictions')
plt.title('Comparing Train and Test MSE Across Different Polynomial Features')

plt.legend()
plt.show()




In [ ]:
# Look at what the 15 degree model looks like on our train data?
degree = 15

# Create the features for the design matrix
expander = PolynomialFeatures(degree=degree, include_bias=False)

# Apply the expander to our train and test data
Z_train = expander.fit_transform(X_train[['miles_scaled']])
Z_test = expander.transform(X_test[['miles_scaled']])  # use transform instead of fit_transform because we want to use the transformation we made on the train data set

# Get the names of these variables
names = expander.get_feature_names_out()

# Create a new, expanded dataframe for the train and test data sets
X_train_design_mat = pd.DataFrame(data=Z_train, columns = names)
X_test_design_mat = pd.DataFrame(data=Z_test, columns = names)

# Fit a linear regression model on the train data design matrix
model = LinearRegression()
model = model.fit(X_train_design_mat, y_train)

# Get predictions on the train and test data sets
y_hat_train = model.predict(X_train_design_mat)
y_hat_test = model.predict(X_test_design_mat)

# Look at the model fit to the training data
plt.scatter(
    X_train_design_mat['miles_scaled'],
    y_train,
    label = 'True Data'
)

X_train_plot = X_train_design_mat.copy()
X_train_plot['y_hat'] = y_hat_train
X_train_plot = X_train_plot.sort_values('miles_scaled')

plt.plot(
    X_train_plot['miles_scaled'],
    X_train_plot['y_hat'],
    color = 'red',
    label = 'Predicted Data'
)
plt.title(f'Training Data and Predictions with Degree {degree}')

plt.show()

# Look at the model applied to the test data set
plt.scatter(
    X_test_design_mat['miles_scaled'],
    y_test,
    label = 'True Data'
)

X_test_plot = X_test_design_mat.copy()
X_test_plot['y_hat'] = y_hat_test
X_test_plot = X_test_plot.sort_values('miles_scaled')

plt.plot(
    X_test_plot['miles_scaled'],
    X_test_plot['y_hat'],
    color = 'red',
    label = 'Predicted Data'
)
plt.title(f'Testing Data and Predictions with Degree {degree}')

plt.show()



### Appendix

In [ ]:
# We can visualize our models in 3D

# Define the interaction variable
X_scaled['miles_x_age'] = X_scaled['miles_scaled'] * X_scaled['age_scaled']

# Look at the data frame of miles_scaled, age_scaled, and mileage_x_age
print('Design Matrix with Interaction Term:\n', X_scaled[['miles_scaled', 'age_scaled', 'miles_x_age']])

# Include it in a model
X = X_scaled[['price', 'miles_scaled', 'age_scaled', 'miles_x_age']]
y = X['price']
X = X.drop(columns = ['price'])

# Fit the model
model = LinearRegression()
model = model.fit(X, y)

# Print the intercept and coefficients
print('Intercept:', model.intercept_)

pd.DataFrame(
    {
        'Variable': model.feature_names_in_,
        'Coefficient': model.coef_
    }
)

# Extract model coefficients and intercept
coefs = model.coef_
inter = model.intercept_

# Create a meshgrid for miles and age
miles_vals = np.linspace(X_scaled['miles_scaled'].min(), X_scaled['miles_scaled'].max(), 50)
age_vals = np.linspace(X_scaled['age_scaled'].min(), X_scaled['age_scaled'].max(), 50)
MILES, AGE = np.meshgrid(miles_vals, age_vals)

# Compute predicted sales (Z-axis) using the regression equation
PREDICTED_PRICE = inter + coefs[0] * MILES + coefs[1] * AGE + coefs[2] * MILES*AGE

# Plot the hyperplane
fig = go.Figure(data=go.Surface(x=MILES, y=AGE, z=PREDICTED_PRICE, colorscale='jet', opacity=0.7))

# Update layout with correct axis labels
fig.update_layout(
    scene=dict(
        xaxis_title="Miles",
        yaxis_title="Age",
        zaxis_title="Price",
        xaxis_range=[X_scaled['miles_scaled'].min(), X_scaled['miles_scaled'].max()],
        yaxis_range=[X_scaled['age_scaled'].min(), X_scaled['age_scaled'].max()],
        zaxis_range=[X_scaled['price'].min(), X_scaled['price'].max()]
    ),
    title="Regression Hyperplane for Price Prediction",
    height=1000,
    width=1000
)

# Add scatter points (actual data)
fig.add_trace(go.Scatter3d(
    x=X_scaled['miles_scaled'],
    y=X_scaled['age_scaled'],
    z=X_scaled['price'],
    mode='markers',
    marker=dict(size=4, color='black', opacity=1)
))

fig.show()